# Exploration : salaires des développeurs en France

Enquête Stack Overflow 2026. Objectif : trouver les constats les plus nets
pour le message du tableau de bord destiné à la direction de la rémunération d'une ESN.

In [ ]:
import pandas as pd

from salaires.chargement import RACINE
from salaires.logique import SEUIL_N
from salaires.population import ajouter_niveau

## Chargement

Je relis le CSV brut directement avec pandas, en ne gardant que les colonnes utiles.

In [ ]:
COLONNES = ["Country", "CodeMain", "Employment", "Currency", "Compensation",
            "WorkExp", "Industry", "WorkSetting"]
brut = pd.read_csv(RACINE / "data" / "raw" / "2026" / "survey_results_public.csv", usecols=COLONNES)
brut.shape

## Population

Filtres décrits dans `src/salaires/population.py`, appliqués un par un
pour voir combien de répondants chacun retire.

In [ ]:
filtres = {
    "France": brut["Country"] == "France",
    "Code comme activité principale": brut["CodeMain"] == "I create code as a primary part of my job",
    "Salarié": brut["Employment"] == "Employed",
    "Salaire déclaré en euros": brut["Currency"] == "EUR European Euro",
    "Salaire entre 15 000 et 200 000 €": brut["Compensation"].between(15_000, 200_000),
}

masque = pd.Series(True, index=brut.index)
etapes = [("Tous les répondants", len(brut))]
for nom, condition in filtres.items():
    masque = masque & condition
    etapes.append((nom, int(masque.sum())))

pop = brut[masque].rename(columns={"Compensation": "SalaireEUR"})
pop = ajouter_niveau(pop)
pd.DataFrame(etapes, columns=["Filtre", "Répondants restants"])